# `customers.csv` 데이터 수치화

- `customers.csv`를 데이터프레임으로 읽습니다.
- 범주형 데이터는 컴퓨터가 읽을 수 있도록 숫자로 변환합니다.
- 마지막에 모든 컬럼이 수치형인지 확인합니다.


In [ ]:
from pathlib import Path

import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]
candidate_paths = []

for root in search_roots:
    candidate_paths.extend([
        root / "customers.csv",
        root / "chapter1_Getting Started" / "customers.csv",
        root / "2026_Spring_Data_Standardization" / "chapter1_Getting Started" / "customers.csv",
    ])

for candidate in candidate_paths:
    if candidate.exists():
        data_path = candidate
        break
else:
    raise FileNotFoundError("customers.csv 파일을 찾을 수 없습니다.")

customers = pd.read_csv(data_path, encoding="utf-8-sig")
print("읽은 파일:", data_path)
customers.head()


## 인코딩 기준

- `고객ID`: `CUST_`를 제거하고 정수형으로 변환
- `회원등급`: 순서형 변수이므로 `Bronze < Silver < Gold < VIP < VVIP` 순서로 숫자 매핑
- `성별`, `결제수단`, `거주지`: 명목형 변수이므로 원-핫 인코딩
- 나머지 수치형 컬럼은 그대로 유지


In [ ]:
grade_map = {
    "Bronze": 1,
    "Silver": 2,
    "Gold": 3,
    "VIP": 4,
    "VVIP": 5,
}

customers_numeric = customers.copy()

customers_numeric["고객ID"] = (
    customers_numeric["고객ID"]
    .str.replace("CUST_", "", regex=False)
    .astype(int)
)

customers_numeric["회원등급"] = customers_numeric["회원등급"].map(grade_map).astype(int)

customers_numeric = pd.get_dummies(
    customers_numeric,
    columns=["성별", "결제수단", "거주지"],
    dtype=int,
)

customers_numeric.head()


In [ ]:
non_numeric_columns = customers_numeric.select_dtypes(exclude="number").columns.tolist()

print("변환 후 데이터 크기:", customers_numeric.shape)
print("숫자가 아닌 컬럼:", non_numeric_columns)
print("모든 컬럼이 수치형인가?:", len(non_numeric_columns) == 0)

customers_numeric.dtypes
